# ЯПИИ 2024
## Лабораторная работа 1
#### Студент: Еленга Невлора Люглеш
#### Ст: 1032245779
#### Группа:НПМмд-01-24
### Задачи:
. Установить tensorflow 2. Создать трансформерную сеть для определения местоположения объектов относительно друг друга.

. Сделать возможность сравнения (как в VQA) местоположения объектов друг с другом.

. Relational reasoning – датасет CLEVR.

. Зарегистрировать GitHub аккаунт

1. Установка tensorflow 2.

In [ ]:
!pip install tensorflow

In [18]:
import tensorflow as tf

- Создать трансформерную сеть для определения местоположения объектов относительно друг друга.

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

1.2 - Предварительная обработка данных .
- Извлечение информации о местоположениях объектов из файлов json.

In [22]:
with open(r'\Users\Лора\Downloads\CLEVR_train_scenes.json','r') as f:
    train_scenes = json.load(f)
with open(r'\Users\Лора\Downloads\CLEVR_train_questions.json','r') as f:
    train_questions = json.load(f)
with open(r'\Users\Лора\Downloads\CLEVR_val_scenes.json','r') as f:
    val_scenes = json.load(f)
with open(r'\Users\Лора\Downloads\CLEVR_val_questions.json','r') as f:
    val_questions = json.load(f)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 0-1: truncated \UXXXXXXXX escape (<ipython-input-22-0a16c5cf5685>, line 1)

In [ ]:
import json

def load_clevr_data(json_path):
    with open(json_path) as f:
        data = json.load(f)
    return data

clevr_data = load_clevr_data('https://cs.stanford.edu/people/jcjohns/clevr/')
#path/to/clevr_dataset.json
# https://cs.stanford.edu/people/jcjohns/clevr/

FileNotFoundError: [Errno 2] No such file or directory: 'https://cs.stanford.edu/people/jcjohns/clevr/'

In [ ]:
def extract_object_positions(data):
    object_positions = []
    for scene in data['scenes']:
        for obj in scene['objects']:
            position = obj['position']  # Предполагаем, что позиции хранятся здесь
            object_positions.append(position)
    return object_positions

positions = extract_object_positions(clevr_data)

3- Предварительная обработка данных

In [ ]:
# Объединение вопросов
questions_a = [q['question'] for q in train_questions['questions'] + val_questions['questions']]

# Инициализация токенизатора и Подгонка токенизатора
question_tokenizer = Tokenizer(oov_token='<OOV>')
question_tokenizer.fit_on_texts(questions_a)

X_train_questions_seq = question_tokenizer.texts_to_sequences([q['question'] for q in train_questions['questions']])
X_val_questions_seq = question_tokenizer.texts_to_sequences([q['question'] for q in val_questions['questions']])

max_question_length = max(len(seq) for seq in X_train_questions_seq + X_val_questions_seq)
X_train_questions_padded = pad_sequences(X_train_questions_seq, maxlen=max_question_length, padding='post')
X_val_questions_padded = pad_sequences(X_val_questions_seq, maxlen=max_question_length, padding='post')


In [ ]:
# Извлечение объектов из сцен
def extract_features(scene):
    features = []
    for obj in scene['objects']:
        attributes = [obj['size'], obj['color'], obj['material'], obj['shape']]
        features.extend(attributes)
    return features

In [ ]:
#  подготовка набора данных в функции
def prep_dataset(scenes, questions):
    X = []
    y = []
    s_dict = {scene['image_index']: scene for scene in scenes['scenes']}
    for question in questions['questions']:
        image_index = question['image_index']
        if image_index in s_dict:
            scene = s_dict[image_index]
            features = extract_features(scene)
            X.append(features)
            y.append(question['answer'])
    return X, y

X_train_scenes_raw, y_train_raw = prep_dataset(train_scenes, train_questions)
X_val_scenes_raw, y_val_raw = prep_dataset(val_scenes, val_questions)

In [ ]:
# кодирование объектов сцены, дополнение объектов сцены для достижения максимальной длины сцены
a_features = [item for sublist in X_train_scenes_raw + X_val_scenes_raw for item in sublist]
scene_encoder = LabelEncoder()
scene_encoder.fit(all_features)

X_train_scenes_encoded = [scene_encoder.transform(features) for features in X_train_scenes_raw]
X_val_scenes_encoded = [scene_encoder.transform(features) for features in X_val_scenes_raw]

max_scene_length = max(len(seq) for seq in X_train_scenes_encoded + X_val_scenes_encoded)
X_train_scenes_padded = pad_sequences(X_train_scenes_encoded, maxlen=max_scene_length, padding='post')
X_val_scenes_padded = pad_sequences(X_val_scenes_encoded, maxlen=max_scene_length, padding='post')

- Кодирование всех слоев.

In [ ]:
a_answers = y_train_raw + y_val_raw

label_encoder = LabelEncoder()
label_encoder.fit(all_answers)

y_train_encoded = label_encoder.transform(y_train_raw)
y_val_encoded = label_encoder.transform(y_val_raw)

4- Модель

In [ ]:
# Определение модели
q_input = tf.keras.layers.Input(shape=(max_question_length,), name='question_input')
q_embedding = tf.keras.layers.Embedding(
    input_dim=len(question_tokenizer.word_index) + 1,
    output_dim=128,
    mask_zero=True
)(q_input)
question_lstm = tf.keras.layers.LSTM(64)(q_embedding)
s_input = tf.keras.layers.Input(shape=(max_scene_length,), name='scene_input')
s_embedding = tf.keras.layers.Embedding(
    input_dim=len(scene_encoder.classes_),
    output_dim=128,
    mask_zero=True
)(s_input)
scene_lstm = tf.keras.layers.LSTM(64)(s_embedding)

combined = tf.keras.layers.concatenate([question_lstm, scene_lstm])


# добавление плотного слоя с помощью ReLU, выходного слоя с плотным слоем с активацией softmax
fc1 = tf.keras.layers.Dense(64, activation='relu')(combined)
output = tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax')(fc1)
model = tf.keras.models.Model(inputs=[q_input, s_input], outputs=output)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

- Создание трансформерной архитектуры

In [ ]:
class TransformerModel(tf.keras.Model):

    def __init__(self, num_layers, d_model, num_heads, dff, input_shape):
        super(TransformerModel, self).__init__()
        self.encoder = tf.keras.layers.Embedding(input_dim=input_shape[0], output_dim=d_model)
        self.transformer_blocks = [tf.keras.layers.TransformerBlock(d_model, num_heads, dff) for _ in range(num_layers)]
        self.final_layer = tf.keras.layers.Dense(1)

    def call(self, x):
        x = self.encoder(x)
        for block in self.transformer_blocks:
            x = block(x)
        return self.final_layer(x)


- Создание функции сравнения

In [ ]:
def compare_positions(pos1, pos2):

    return np.linalg.norm(pos1 - pos2)

5- обучение модели

In [ ]:
history = model.fit(
    {'question_input': X_train_questions_padded, 'scene_input': X_train_scenes_padded},
    y_train_encoded,
    epochs=10,
    batch_size=32,
    validation_data=(
        {'question_input': X_val_questions_padded, 'scene_input': X_val_scenes_padded},
        y_val_encoded
    )
)

In [ ]:
model = TransformerModel(num_layers=4, d_model=128, num_heads=8, dff=512, input_shape=(images.shape[1], images.shape[2]))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(images, answers, epochs=10, validation_split=0.2)

In [ ]:
predictions = model.predict(images)
# Сравнение позиций
comparison_result = compare_positions(predictions[0], predictions[1])

6- Построение графика train & validation : accuracy & loss.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='T-Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='V-Accuracy', marker='o')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Acc')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='T-Loss', marker='o')
plt.plot(history.history['val_loss'], label='V-Loss', marker='o')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

7- Визуализация прогнозов

In [ ]:
indices = np.random.choice(len(X_val_questions_padded), 5, replace=False)

test_questions = X_val_questions_padded[indices]
test_scenes = X_val_scenes_padded[indices]
test_labels = y_val_encoded[indices]

predict = model.predict({'question_input': test_questions, 'scene_input': test_scenes})
predicted = np.argmax(predict, axis=1)

for i, idx in enumerate(indices):
    question_text = val_questions['questions'][idx]['question']
    true_answer = label_encoder.inverse_transform([test_labels[i]])[0]
    pred_answer = label_encoder.inverse_transform([predicted[i]])[0]
    print(f"--------------------------------------------------------------------------\nQuestion: {question_text}\nTrue Answer: {true_answer}\nPredicted Answer: {pred_answer}\n--------------------------------------------------------------------------\n")